# Uncertainty-aware Atomistic Modeling with SchNet and Bayesian Neural Networks

This tutorial demonstrates a complete workflow for training an uncertainty-aware
atomistic machine learning potential by combining:

- **SchNet** atomic representation
- **Bayesian Neural Network (BNN)** energy prediction head
- **Uncertainty estimation**
- **t-SNE feature extraction**
- **Uncertainty-driven active learning selection**

The workflow follows the style of the SchNetPack QM9 tutorial and is designed
for atomic cluster datasets.


## 1. Import packages

We use SchNetPack for atomistic neural network modeling,
PyTorch Lightning for training, and custom Bayesian modules for uncertainty
quantification.

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import torch
import torchmetrics
import pytorch_lightning as pl

import schnetpack as spk
import schnetpack.transform as trn

from schnetpack.utils import load_model
from sklearn.metrics import pairwise_distances

import allnn
import integration


## 2. Load configuration

All paths and hyperparameters are controlled through `config.yaml`.

This makes the workflow reproducible and avoids hard-coded parameters.

In [ ]:
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

dataset_dir = cfg["paths"]["dataset_dir"]
output_dir = cfg["paths"]["output_dir"]
model_dir = cfg["paths"]["model_dir"]

batch_size = cfg["data"]["batch_size"]
cutoff = cfg["model"]["cutoff"]

os.makedirs(output_dir, exist_ok=True)


## 3. Load atomic datasets

The dataset follows the ASEAtomsData format used by SchNetPack.

Energy is used as the regression target.
Neighbor lists are generated using a cutoff radius.

In [ ]:
train_data = spk.data.AtomsDataModule(
    f"{dataset_dir}/{cfg['data']['train_db']}",
    batch_size=batch_size,
    distance_unit="Ang",
    property_units={"energy_U0": "eV"},
    num_train=0.9,
    num_val=0.1,
    split_file=cfg["data"]["split_train"],
    transforms=[
        trn.ASENeighborList(cutoff=cutoff),
        trn.RemoveOffsets("energy_U0", remove_mean=True),
        trn.CastTo32(),
    ],
)

train_data.prepare_data()
train_data.setup()

print("Training structures:", len(train_data.train_dataset))
print("Validation structures:", len(train_data.val_dataset))


## 4. Construct SchNet representation

SchNet extracts atom-centered environmental features using continuous-filter
convolutions.

In [ ]:
pairwise = spk.atomistic.PairwiseDistances()

rbf = spk.nn.GaussianRBF(
    n_rbf=cfg["model"]["n_rbf"],
    cutoff=cutoff
)

schnet = spk.representation.SchNet(
    n_atom_basis=cfg["model"]["n_atom_basis"],
    n_interactions=1,
    radial_basis=rbf,
    cutoff_fn=spk.nn.CosineCutoff(cutoff),
)


## 5. Bayesian Neural Network output module

The BNN predicts both energy and model uncertainty.
The uncertainty is obtained from the predictive distribution.

In [ ]:
pred = allnn.BayesianNN(
    n_in=cfg["model"]["n_atom_basis"],
    output_key="energy_U0",
)

model = spk.model.NeuralNetworkPotential(
    representation=schnet,
    input_modules=[pairwise],
    output_modules=pred,
    postprocessors=[
        trn.CastTo64(),
        trn.AddOffsets("energy_U0", add_mean=True),
    ],
)


## 6. Define training task

In [ ]:
output = integration.ModelOutput_BNN(
    name="energy_U0",
    loss_fn=torch.nn.MSELoss(),
    loss_weight=1.0,
    metrics={
        "MAE": torchmetrics.MeanAbsoluteError(),
        "MSE": torchmetrics.MeanSquaredError(),
    },
)

task = integration.AtomisticTask_BNN(
    model=model,
    outputs=[output],
    optimizer_cls=torch.optim.AdamW,
    optimizer_args={
        "lr": cfg["model"]["lr"]
    },
)


## 7. Train model

The best checkpoint is selected according to validation loss.

In [ ]:
trainer = pl.Trainer(
    max_epochs=cfg["model"]["max_epochs"],
    default_root_dir=output_dir,
    logger=pl.loggers.CSVLogger(
        output_dir,
        name="training"
    ),
    callbacks=[
        spk.train.ModelCheckpoint(
            model_path=f"{model_dir}/{cfg['files']['checkpoint']}",
            save_top_k=1,
            monitor="val_loss",
        )
    ],
)

trainer.fit(
    task,
    datamodule=train_data
)


## 8. Prediction and uncertainty estimation

After training, the model is used to predict candidate structures.
The cached uncertainty values are extracted from the Bayesian output module.

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = load_model(
    f"{model_dir}/{cfg['files']['checkpoint']}",
    device=device,
)

model.eval()

energies_all = []
unc_all = []

for batch in train_data.train_dataloader():

    batch = {
        k: v.to(device) if hasattr(v, "to") else v
        for k, v in batch.items()
    }

    with torch.no_grad():
        result = model(batch)

    energies_all.append(
        result["energy_U0"].cpu().numpy()
    )

    uncertainty = (
        model.output_modules
        .get_cached_uncertainty()["std"]
        .cpu()
        .numpy()
    )

    unc_all.append(uncertainty)


energies_all = np.concatenate(energies_all)
unc_all = np.concatenate(unc_all)

print(energies_all.shape)
print(unc_all.shape)


## 9. Save prediction results

In [ ]:
pd.DataFrame({
    "Energy": energies_all,
    "Uncertainty": unc_all
}).to_csv(
    "prediction_uncertainty.csv",
    index=False
)


## 10. Active learning selection

Candidate structures are selected using two criteria:

1. High predictive uncertainty
2. Large structural diversity

This avoids selecting only redundant configurations.

In [ ]:
threshold = np.percentile(
    unc_all,
    80
)

uncertain_idx = np.where(
    unc_all > threshold
)[0]

print(
    "High uncertainty samples:",
    len(uncertain_idx)
)


## Summary

This tutorial provides a complete uncertainty-aware workflow:

```
Atomic structures
        |
        v
SchNet representation
        |
        v
Bayesian Neural Network
        |
        +----------------+
        |                |
Energy prediction   Uncertainty estimation
        |
        v
Active learning sampling
```

The approach enables efficient exploration of complex cluster potential
energy surfaces.
